In [1]:
import os
import sys

# 1. Force install libraries inside the active notebook kernel path
!{sys.executable} -m pip install xgboost shap

# 2. Shift python path up one level to see the project root 'src/' directory
sys.path.append(os.path.abspath(os.path.join('..')))

print("✅ [SYSTEM] Dependencies installed and path mappings configured successfully.")

✅ [SYSTEM] Dependencies installed and path mappings configured successfully.


In [6]:
import pandas as pd
from src.modeling import train_risk_models, calculate_risk_premium
from src.interpretability import generate_model_explanations

# Load processed DVC data repository reference
data_path = '../data/processed/cleaned_insurance_data.csv' 
df = pd.read_csv(data_path)

print(f"📊 Dataset successfully loaded. Shape: {df.shape}")

📊 Dataset successfully loaded. Shape: (10000, 21)


In [9]:
# Train severity (regression) and probability (classification) engines
# The modeling package automatically filters columns and extracts continuous features
severity_model, probability_model, feature_cols = train_risk_models(df)

print("\n✨ Selected Predictor Features Used for Training Matrix:")
print(feature_cols)

🚀 [SYSTEM] Initializing Task 4 Statistical Modeling Loop...

📊 Running Severity Benchmarking (Subset where PastClaims > 0)...
   -> Linear Regression | RMSE: 0.62, R2 Score: 0.4594
   -> Random Forest | RMSE: 0.56, R2 Score: 0.5631
   -> XGBoost Regressor | RMSE: 0.61, R2 Score: 0.4734

📊 Running Probability Benchmarking...
   -> Logistic Regression | LogLoss Score: 0.3886, Accuracy: 0.8490
   -> Random Forest Classifier | LogLoss Score: 0.2895, Accuracy: 0.9025
   -> XGBoost Classifier | LogLoss Score: 0.2898, Accuracy: 0.8930

✨ Selected Predictor Features Used for Training Matrix:
['Age', 'AnnualIncome', 'RiskScore', 'Deductible', 'NCD', 'ClaimAmount', 'TotalPremium', 'TotalClaims', 'CustomValueEstimate', 'ZipCode', 'VehicleAge']


In [10]:
# Calculate premiums using a 150 ZAR expense loading and a 15% target profit margin
priced_df = calculate_risk_premium(
    df=df, 
    severity_model=severity_model, 
    probability_model=probability_model, 
    features=feature_cols,
    expense_loading=150.0,
    profit_margin=0.15
)

# Discover the premium column names dynamically to display old vs. new configurations
old_premium_col = [c for c in df.columns if 'premium' in c.lower()][0]

# View validation output mapping
print("\n📋 Comparison Sample: Historical Premium vs. Newly Calculated Risk Premium:")
print(priced_df[[old_premium_col, 'Calculated_Risk_Premium']].dropna().head(10))


✅ [SUCCESS] Risk-based Premium engine optimized across portfolio variables.

📋 Comparison Sample: Historical Premium vs. Newly Calculated Risk Premium:
   AnnualPremium  Calculated_Risk_Premium
0           2346               177.785004
1           2334               181.251007
2           1697               177.604279
3           2370               176.472626
4           2582               181.192230
5           1310               177.614944
6           2204               176.510147
7           1590               176.471100
8           2665               177.642883
9           2527               180.013870


In [11]:
# Pass the champion severity engine to generate feature impact visualizations
generate_model_explanations(severity_model, priced_df, feature_cols)


🚀 [SYSTEM] Executing Model Interpretability Framework (SHAP Engine)...
[SUCCESS] Exported SHAP explainability charts to: ../reports/figures/05_shap_feature_importance.png
